In [20]:
import csv
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

# ===== SET THESE =====
db_path = "FVEDB.db"
csv_path = "Ibehej.csv"
table_name = "MqttData"

# DB column names
date_col = "Date"   # dd.MM.yyyy
time_col = "Time"     # HH:mm:ss

# CSV timestamp column
csv_timestamp_col = "TimeStamp"

# Matching tolerance
tolerance_minutes = 5

# True = only preview
# False = really insert
dry_run = False

In [19]:
import csv
import sqlite3
from bisect import bisect_left
from datetime import datetime, timedelta
from pathlib import Path


def parse_csv_timestamp(value: str) -> datetime:
    return datetime.strptime(value.strip(), "%Y-%m-%d %H:%M:%S")


def split_to_db_date_time(dt: datetime) -> tuple[str, str]:
    return dt.strftime("%d.%m.%Y"), dt.strftime("%H:%M:%S")


def parse_db_datetime(date_str, time_str):
    if date_str is None or time_str is None:
        return None
    try:
        return datetime.strptime(
            f"{str(date_str).strip()} {str(time_str).strip()}",
            "%d.%m.%Y %H:%M:%S"
        )
    except ValueError:
        return None


def get_table_columns(conn, table_name):
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    if not rows:
        raise RuntimeError(f"Table '{table_name}' not found.")
    return [row[1] for row in rows]


def load_existing_db_data(conn, table_name, date_col, time_col):
    rows = conn.execute(f"SELECT {date_col}, {time_col} FROM {table_name}").fetchall()

    exact_keys = set()
    datetimes = []

    for d, t in rows:
        if d is None or t is None:
            continue
        d = str(d).strip()
        t = str(t).strip()
        exact_keys.add((d, t))

        dt = parse_db_datetime(d, t)
        if dt is not None:
            datetimes.append(dt)

    datetimes.sort()
    return exact_keys, datetimes


def has_existing_within_tolerance(target_dt, sorted_datetimes, tolerance):
    left = bisect_left(sorted_datetimes, target_dt - tolerance)

    while left < len(sorted_datetimes):
        dt = sorted_datetimes[left]
        if dt > target_dt + tolerance:
            break
        return True
    return False


def parse_value(value):
    if value is None:
        return None

    value = value.strip()
    if value == "":
        return None

    try:
        if value.isdigit() or (value.startswith("-") and value[1:].isdigit()):
            return int(value)
    except Exception:
        pass

    try:
        return float(value)
    except ValueError:
        return value


def collect_rows_to_insert(
    csv_path,
    existing_exact_keys,
    existing_datetimes,
    csv_timestamp_col,
    date_col,
    time_col,
    tolerance_minutes=5,
):
    tolerance = timedelta(minutes=tolerance_minutes)
    rows_to_insert = []

    # prevents duplicate exact timestamps inside CSV from being inserted twice
    planned_exact_keys = set(existing_exact_keys)

    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")

        if reader.fieldnames is None:
            raise RuntimeError("CSV header not found.")

        if csv_timestamp_col not in reader.fieldnames:
            raise RuntimeError(f"CSV does not contain '{csv_timestamp_col}' column.")

        for row in reader:
            csv_dt = parse_csv_timestamp(row[csv_timestamp_col])
            db_date, db_time = split_to_db_date_time(csv_dt)
            exact_key = (db_date, db_time)

            # skip if exact Date+Time already exists in DB or was already selected from CSV
            if exact_key in planned_exact_keys:
                continue

            # skip if DB already has something within +- tolerance
            if has_existing_within_tolerance(csv_dt, existing_datetimes, tolerance):
                continue

            row[date_col] = db_date
            row[time_col] = db_time
            rows_to_insert.append(row)
            planned_exact_keys.add(exact_key)

    return rows_to_insert


def insert_rows_insert_only(conn, table_name, rows, table_columns, csv_timestamp_col):
    if not rows:
        return 0, []

    ignore_columns = {
        csv_timestamp_col,
        "ID_ZDROJE",
        "ID_zaznamu",
    }

    insert_columns = [
        col for col in rows[0].keys()
        if col in table_columns and col not in ignore_columns
    ]

    if not insert_columns:
        raise RuntimeError("No matching insertable columns found.")

    placeholders = ", ".join(["?"] * len(insert_columns))
    columns_sql = ", ".join(insert_columns)
    sql = f"INSERT INTO {table_name} ({columns_sql}) VALUES ({placeholders})"

    inserted = 0

    with conn:
        for row in rows:
            values = [parse_value(row[col]) for col in insert_columns]

            # plain INSERT only
            conn.execute(sql, values)
            inserted += 1

    return inserted, insert_columns

In [21]:

if not Path(db_path).exists():
    raise FileNotFoundError(db_path)

if not Path(csv_path).exists():
    raise FileNotFoundError(csv_path)

conn = sqlite3.connect(db_path)

try:
    table_columns = get_table_columns(conn, table_name)
    print("Table columns:", table_columns)

    if date_col not in table_columns:
        raise RuntimeError(f"Column '{date_col}' not found in table.")
    if time_col not in table_columns:
        raise RuntimeError(f"Column '{time_col}' not found in table.")

    existing_exact_keys, existing_datetimes = load_existing_db_data(
        conn, table_name, date_col, time_col
    )

    print("Existing exact DB timestamps:", len(existing_exact_keys))

    rows_to_insert = collect_rows_to_insert(
        csv_path=csv_path,
        existing_exact_keys=existing_exact_keys,
        existing_datetimes=existing_datetimes,
        csv_timestamp_col=csv_timestamp_col,
        date_col=date_col,
        time_col=time_col,
        tolerance_minutes=tolerance_minutes,
    )

    print(f"Rows to insert: {len(rows_to_insert)}")

    if rows_to_insert:
        print("\nFirst 10 rows to insert:")
        for row in rows_to_insert[:10]:
            print(row[date_col], row[time_col])

    if dry_run:
        print("\nDry run only. No database changes were made.")
    else:
        inserted_count, used_columns = insert_rows_insert_only(
            conn=conn,
            table_name=table_name,
            rows=rows_to_insert,
            table_columns=table_columns,
            csv_timestamp_col=csv_timestamp_col,
        )
        print(f"\nInserted rows: {inserted_count}")
        print("Inserted columns:", used_columns)

finally:
    conn.close()

Table columns: ['Date', 'Time', 'BUY', 'BUY_T', 'Charge', 'Consumption', 'DateTime', 'Discharge', 'Feed', 'FromBAT', 'FromBAT_T', 'FullMessage', 'GRID_LIMIT', 'Input', 'Load', 'Output', 'PRICE_CZK', 'PVenergy', 'PVenergy_T', 'P_BAT', 'P_Batt', 'P_EPS', 'P_GRID', 'P_GRID_L1', 'P_GRID_L2', 'P_GRID_L3', 'P_HOME', 'P_HOME_L1', 'P_HOME_L2', 'P_HOME_L3', 'P_Inv', 'P_OffGr', 'P_OnGr', 'P_PV', 'SELL', 'SELL_T', 'SoC', 'ToBAT', 'ToBAT_T']
Existing exact DB timestamps: 123754
Rows to insert: 26731

First 10 rows to insert:
20.05.2025 16:40:00
20.05.2025 16:45:00
20.05.2025 16:50:00
20.05.2025 16:55:00
20.05.2025 17:00:00
20.05.2025 17:05:00
20.05.2025 17:10:00
20.05.2025 17:15:00
20.05.2025 17:20:00
20.05.2025 17:25:00

Inserted rows: 26731
Inserted columns: ['SoC', 'P_PV', 'P_HOME', 'P_HOME_L1', 'P_HOME_L2', 'P_HOME_L3', 'P_EPS', 'P_GRID', 'P_GRID_L1', 'P_GRID_L2', 'P_GRID_L3', 'P_BAT', 'PVenergy', 'PVenergy_T', 'ToBAT', 'ToBAT_T', 'FromBAT', 'FromBAT_T', 'SELL', 'SELL_T', 'BUY', 'BUY_T', 'GRID

In [16]:
import shutil
shutil.copy2(db_path, db_path + ".backup")
print("Backup created:", db_path + ".backup")

Backup created: FVEDB.db.backup


In [8]:
import pandas as pd

pd.DataFrame(missing_rows[:20])

,ID_zaznamu,ID_ZDROJE,TimeStamp,SoC,P_PV,P_HOME,P_HOME_L1,P_HOME_L2,P_HOME_L3,P_EPS,...,FromBAT_T,SELL,SELL_T,BUY,BUY_T,Consumed,Consumed_T,GRID_LIMIT,Date,Time
0,5,5,2025-05-20 16:40:00,95.0,1711.0,-382.666666666667,-79.6666666666667,-95.0,-208.0,0.0,...,3116.5,3.53666666666667,1399.34666666667,8.5,11034.6,22.8633333333333,15998.2533333333,,20.05.2025,16:40:00
1,7,5,2025-05-20 16:45:00,95.0,1409.73913043478,-370.0,-76.3913043478261,-80.6086956521739,-213.0,0.0,...,3116.5,3.54,1399.35,8.5,11034.6,22.9686956521739,15998.3586956522,,20.05.2025,16:45:00
2,12,5,2025-05-20 16:50:00,94.7586206896552,1577.62068965517,-356.793103448276,-71.1379310344828,-85.2068965517241,-200.448275862069,0.0,...,3116.5,3.54,1399.35,8.5,11034.6,23.0462068965517,15998.4362068966,,20.05.2025,16:50:00
3,17,5,2025-05-20 16:55:00,94.7,2294.16666666667,-358.6,-71.0,-85.4,-202.2,0.0,...,3116.59666666667,3.54,1399.35,8.5,11034.6,23.2433333333333,15998.6333333333,,20.05.2025,16:55:00
4,22,5,2025-05-20 17:00:00,95.0,4535.79310344828,-525.0,-145.48275862069,-147.965517241379,-231.551724137931,0.0,...,3116.6,3.54,1399.35,8.5,11034.6,23.3806896551724,15998.7706896552,,20.05.2025,17:00:00
5,27,5,2025-05-20 17:05:00,95.0,4539.4,-542.066666666667,-149.266666666667,-168.466666666667,-224.333333333333,0.0,...,3116.6,3.54533333333333,1399.35533333333,8.5,11034.6,23.5213333333333,15998.8846666667,,20.05.2025,17:05:00
6,32,5,2025-05-20 17:10:00,99.5862068965517,3858.48275862069,-2213.72413793103,-719.344827586207,-798.931034482759,-695.448275862069,0.0,...,3116.6,3.65413793103448,1399.46413793103,8.5,11034.6,23.6389655172414,15999.0289655172,,20.05.2025,17:10:00
7,37,5,2025-05-20 17:15:00,100.0,3619.46666666667,-2571.6,-901.6,-794.333333333333,-875.666666666667,0.0,...,3116.6,3.859,1399.669,8.5,11034.6,23.7643333333333,15999.1543333333,,20.05.2025,17:15:00
8,42,5,2025-05-20 17:20:00,100.0,1719.03448275862,-722.689655172414,-222.034482758621,-203.896551724138,-296.758620689655,0.0,...,3116.6,3.98137931034483,1399.79137931034,8.5,11034.6,23.86,15999.25,,20.05.2025,17:20:00
9,47,5,2025-05-20 17:25:00,99.4,903.066666666667,-466.666666666667,-90.2,-141.266666666667,-235.2,0.0,...,3116.6,4.01866666666666,1399.82866666667,8.5,11034.6,23.928,15999.318,,20.05.2025,17:25:00
